# ALM Data Pipeline Tutorial

This notebook walks through the ALM (Audio Language Model) data pipeline:
1. Read diarized audio manifests
2. Build training windows with quality filters
3. Remove overlapping windows
4. Inspect and visualize results

**No GPU required** — all stages are CPU-only. Install with `uv sync --extra audio_cpu`.

In [ ]:
import json
import os

from nemo_curator.backends.xenna import XennaExecutor
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.audio.alm.alm_data_builder import ALMDataBuilderStage
from nemo_curator.stages.audio.alm.alm_data_overlap import ALMDataOverlapStage
from nemo_curator.stages.audio.alm.alm_manifest_reader import ALMManifestReaderStage
from nemo_curator.stages.audio.alm.alm_manifest_writer import ALMManifestWriterStage

## Step 1: Inspect the input data

The bundled sample has 5 audio manifest entries with diarized segments.

In [ ]:
MANIFEST_PATH = "tests/fixtures/audio/alm/sample_input.jsonl"
OUTPUT_DIR = "./alm_tutorial_output"

with open(MANIFEST_PATH) as f:
    entries = [json.loads(line) for line in f if line.strip()]

print(f"Input entries: {len(entries)}")
for i, entry in enumerate(entries):
    n_segments = len(entry.get("segments", []))
    sr = entry.get("audio_sample_rate", "unknown")
    speakers = len({s.get("speaker", "") for s in entry.get("segments", [])})
    print(f"  [{i}] {n_segments} segments, {speakers} speakers, {sr} Hz")

## Step 2: Run the ALM pipeline

The pipeline reads manifests, builds windows, filters overlaps, and writes output.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, "alm_output.jsonl")

pipeline = Pipeline(
    name="alm_tutorial",
    description="Build ALM training windows from diarized manifests",
)

pipeline.add_stage(ALMManifestReaderStage(manifest_paths=[MANIFEST_PATH]))
pipeline.add_stage(
    ALMDataBuilderStage(
        target_window_duration=120.0,
        tolerance=0.1,
        min_sample_rate=16000,
        min_bandwidth=8000,
        min_speakers=2,
        max_speakers=5,
    )
)
pipeline.add_stage(ALMDataOverlapStage(overlap_percentage=50))
pipeline.add_stage(ALMManifestWriterStage(output_path=output_path))

print(pipeline.describe())

In [ ]:
executor = XennaExecutor()
pipeline.run(executor)
print("Pipeline complete!")

## Step 3: Inspect results

In [ ]:
with open(output_path) as f:
    results = [json.loads(line) for line in f if line.strip()]

print(f"Output entries: {len(results)}")

total_windows_before = sum(len(r.get("windows", [])) for r in results)
total_windows_after = sum(len(r.get("filtered_windows", [])) for r in results)
total_duration = sum(r.get("filtered_dur", 0) for r in results)

print(f"Windows before overlap filter: {total_windows_before}")
print(f"Windows after overlap filter:  {total_windows_after}")
print(f"Total filtered audio duration: {total_duration:.1f}s ({total_duration / 3600:.2f}h)")
print(
    f"Overlap reduction: {(1 - total_windows_after / total_windows_before) * 100:.0f}%"
    if total_windows_before
    else "N/A"
)

## Step 4: Understand the filtering statistics

In [ ]:
for i, r in enumerate(results):
    stats = r.get("stats", {})
    print(f"Entry [{i}]:")
    print(f"  Total segments: {stats.get('total_segments', 'N/A')}")
    print(f"  Total duration: {stats.get('total_dur', 0):.1f}s")
    print(f"  Lost (low bandwidth):    {stats.get('lost_bw', 0)}")
    print(f"  Lost (low sample rate):  {stats.get('lost_sr', 0)}")
    print(f"  Lost (speaker count):    {stats.get('lost_spk', 0)}")
    print(f"  Lost (window constraints): {stats.get('lost_win', 0)}")
    print(f"  Truncation events: {r.get('truncation_events', 0)}")
    print()

## Step 5: Visualize results

### Window duration distribution and filter effects

In [ ]:
# TODO(Sarah): Add matplotlib/seaborn visualizations
# - Window duration histogram (before vs after overlap filter)
# - Speaker count distribution across windows
# - Filter loss breakdown (stacked bar: bandwidth, sample rate, speaker, window)
# - Per-entry window yield bar chart

durations_per_entry = [r.get("filtered_dur_list", []) for r in results]
all_durations = [d for dlist in durations_per_entry for d in dlist]

if all_durations:
    print(
        f"Window durations — min: {min(all_durations):.1f}s, max: {max(all_durations):.1f}s, "
        f"mean: {sum(all_durations) / len(all_durations):.1f}s"
    )
else:
    print("No filtered windows produced.")

## Step 6: Experiment with parameters

Try different overlap thresholds and see how they affect the output:

In [ ]:
from nemo_curator.tasks.audio_task import AudioTask

sample_entry = entries[0]
task = AudioTask(data=sample_entry)

builder = ALMDataBuilderStage(
    target_window_duration=120.0,
    tolerance=0.1,
    min_sample_rate=16000,
    min_bandwidth=8000,
    min_speakers=2,
    max_speakers=5,
)
built = builder.process(task)
n_windows = len(built.data.get("windows", []))

print(f"Windows from entry[0]: {n_windows}")
print()

for pct in [0, 25, 50, 75, 100]:
    overlap = ALMDataOverlapStage(overlap_percentage=pct)
    filtered = overlap.process(built)
    n_filtered = len(filtered.data.get("filtered_windows", []))
    print(
        f"  overlap_percentage={pct:3d}: {n_filtered:3d} windows kept ({n_filtered / n_windows * 100:.0f}%)"
        if n_windows
        else "  No windows"
    )